# Wire Inspector — Google Colab
Upload a wire photo, highlight possible burnmarks, cuts, cracks and scratches, and download an overlay, mask and JSON report.

**Run cells 1–6 in order.** Only this notebook is needed; the inspection engine is included below. No API key is required. An optional GPU runtime speeds up inference; CPU also works. The first model download is several hundred MB.

Photos are uploaded to your Colab runtime and processed there. No external inference API is used. Download results before the runtime ends.

**Experimental:** CLIPSeg was not trained specifically for these defects. Masks are candidate regions, activations are not calibrated probabilities, and a “Normal candidate” is not a quality pass.

In [ ]:
#@title 1. Install dependencies (run first in a fresh runtime)
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.46.3', 'torch>=2.6,<3', 'numpy>=1.26,<3',
    'Pillow>=10.4,<13', 'opencv-python-headless>=4.10,<5',
    'safetensors>=0.4.5', 'pandas', 'ipython'])
print('Dependencies installed. Continue to cell 2.')

In [ ]:
#@title 2. Define the inspection engine
"""One pretrained model; no external inference API or fabricated detections."""
import io
import numpy as np
from PIL import Image, ImageOps

MODEL_ID = 'CIDAS/clipseg-rd64-refined'
LABELS = ['Burnmark', 'Cut', 'Crack', 'Scratch']
PROMPTS = [
    'a burnt scorched patch on electrical wire insulation',
    'a cut opening in electrical wire insulation',
    'a crack or split in electrical wire insulation',
    'a scratch abrasion on electrical wire insulation',
    'an electrical wire or cable',
    'normal intact undamaged electrical wire insulation',
]
COLORS = [(240, 85, 60), (255, 185, 40), (170, 100, 255), (40, 190, 230)]
DESCRIPTIONS = [
    'Possible heat-damaged or scorched patch on the wire surface.',
    'Possible cut or opening in the wire surface.',
    'Possible narrow crack or split in the wire surface.',
    'Possible scratch or abrasion on the wire surface.',
]

def read_image(data):
    image = Image.open(io.BytesIO(data))
    if image.width * image.height > 20_000_000:
        raise ValueError('Please upload an image smaller than 20 megapixels.')
    image = ImageOps.exif_transpose(image).convert('RGB')
    image.thumbnail((1600, 1600))
    return image

class Inspector:
    def __init__(self):
        import torch
        from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.processor = CLIPSegProcessor.from_pretrained(MODEL_ID)
        self.model = CLIPSegForImageSegmentation.from_pretrained(
            MODEL_ID
        ).to(self.device).eval()

    def predict(self, image):
        import torch
        # Sequential prompts keep peak CPU/GPU memory lower than a six-image batch.
        maps = []
        with torch.inference_mode():
            for prompt in PROMPTS:
                inputs = self.processor(text=[prompt], images=[image],
                                        return_tensors='pt')
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                logits = self.model(**inputs, interpolate_pos_encoding=True).logits
                logits = logits.reshape(1, 1, *logits.shape[-2:])
                resized = torch.nn.functional.interpolate(
                    logits, size=(image.height, image.width),
                    mode='bilinear', align_corners=False)
                maps.append(resized.sigmoid()[0, 0].cpu().numpy())
        return np.stack(maps)

def summarize(image, maps, threshold=0.5, wire_threshold=0.35, min_pixels=30):
    """Heuristic candidate masks, never calibrated probabilities or quality decisions."""
    if maps.shape != (6, image.height, image.width):
        raise ValueError('Prediction dimensions do not match the image.')
    wire = maps[4] >= wire_threshold
    mask_ids = np.zeros(wire.shape, dtype=np.uint8)
    rows = []
    best = maps[:4].argmax(axis=0)
    for i, label in enumerate(LABELS):
        mask = wire & (maps[i] >= threshold) & (best == i)
        # Remove isolated pixel noise without smoothing away thin cracks.
        import cv2
        n, components, stats, _ = cv2.connectedComponentsWithStats(mask.astype('uint8'), 8)
        keep = np.zeros(mask.shape, dtype=bool)
        for component in range(1, n):
            if stats[component, cv2.CC_STAT_AREA] >= min_pixels:
                keep |= components == component
        if keep.any():
            mask_ids[keep] = i + 1
            ys, xs = np.where(keep)
            rows.append({'defect': label, 'description': DESCRIPTIONS[i],
                         'mean_activation': round(float(maps[i][keep].mean()), 3),
                         'image_area_percent': round(100 * float(keep.mean()), 2),
                         'bounding_box_xyxy': [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]})
    if wire.sum() < max(min_pixels, wire.size * 0.001):
        status = 'Unable to inspect — wire not reliably localized'
        rows, mask_ids = [], np.zeros_like(mask_ids)
    elif rows:
        status = 'Possible defect(s) — review highlighted regions'
    elif np.mean(maps[5][wire] >= threshold) >= 0.5:
        status = 'Normal candidate — no visible defect detected at these settings'
    else:
        status = 'Uncertain — no defect region passed the selected thresholds'
    overlay = np.array(image).copy()
    for i, color in enumerate(COLORS):
        region = mask_ids == i + 1
        overlay[region] = (overlay[region] * 0.5 + np.array(color) * 0.5).astype('uint8')
    return status, rows, Image.fromarray(overlay), Image.fromarray(mask_ids)


def png_bytes(image):
    output = io.BytesIO()
    image.save(output, format='PNG')
    return output.getvalue()


In [ ]:
#@title 3. Load the pretrained model (cached for this session)
import time
import json
import zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display

if 'inspector' not in globals():
    inspector = Inspector()
print(f'Model ready: {MODEL_ID} | Device: {inspector.device.upper()}')

In [ ]:
#@title 4. Upload one wire image and analyze it
from google.colab import files

# Clear previous results before a new upload, including after failed analysis.
image = maps = report = result_zip = None
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Choose exactly one JPG, PNG or WebP image, then rerun this cell.')
filename, raw = next(iter(uploaded.items()))
image = read_image(raw)
print(f'Analyzing {filename}: {image.width} × {image.height} pixels…')
started = time.perf_counter()
maps = inspector.predict(image)
elapsed = time.perf_counter() - started
assert maps.shape == (6, image.height, image.width) and np.isfinite(maps).all()
print(f'Analysis complete in {elapsed:.1f} seconds. Run cell 5 to view the results.')

In [ ]:
#@title 5. View results — adjust settings and rerun without repeating inference
#@markdown These thresholds are exploratory, not validated quality criteria.
defect_threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
wire_threshold = 0.35 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
minimum_region_pixels = 30 #@param {type:"integer"}
show_activation_maps = False #@param {type:"boolean"}

report = result_zip = None
if image is None or maps is None:
    raise RuntimeError('Complete the image upload and analysis in cell 4 first.')
if not (0.1 <= defect_threshold <= 0.9 and 0.1 <= wire_threshold <= 0.9):
    raise ValueError('Thresholds must be between 0.1 and 0.9.')
if minimum_region_pixels < 1:
    raise ValueError('Minimum region size must be at least 1 pixel.')

status, rows, overlay, mask = summarize(
    image, maps, defect_threshold, wire_threshold, int(minimum_region_pixels))
print(status)
print(f'Inference: {elapsed:.1f}s | {inspector.device.upper()} | {image.width} × {image.height}')
print('Original image')
display(image)
print('Overlay: red = burnmark, yellow = cut, purple = crack, blue = scratch')
display(overlay)
if rows:
    display(pd.DataFrame(rows))
else:
    print('No highlighted defect regions. This does not establish that the wire is defect-free.')
if show_activation_maps:
    for label, activation in zip(LABELS + ['Wire region', 'Intact wire'], maps):
        print(label)
        display(Image.fromarray((np.clip(activation, 0, 1) * 255).astype('uint8')))

report = {
    'model': MODEL_ID, 'source_filename': filename, 'status': status, 'results': rows,
    'processed_size': [image.width, image.height],
    'inference_seconds': round(elapsed, 3), 'device': inspector.device,
    'settings': {'defect_threshold': defect_threshold, 'wire_threshold': wire_threshold,
                 'minimum_region_pixels': int(minimum_region_pixels)},
    'mask_values': {'0': 'background/unassigned', '1': 'Burnmark', '2': 'Cut',
                    '3': 'Crack', '4': 'Scratch'},
    'note': 'Unvalidated zero-shot prototype. Activations are not calibrated probabilities. Mask coordinates refer to the processed image.'
}
# Build the ZIP immediately so it always matches the displayed results.
archive_data = io.BytesIO()
with zipfile.ZipFile(archive_data, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('overlay.png', png_bytes(overlay))
    archive.writestr('mask.png', png_bytes(mask))
    archive.writestr('report.json', json.dumps(report, indent=2))
result_zip = Path('wire_inspection_results.zip')
result_zip.write_bytes(archive_data.getvalue())
print('Results ZIP ready. Run cell 6 to download it.')

In [ ]:
#@title 6. Download the displayed results
if image is None or maps is None or globals().get('report') is None or globals().get('result_zip') is None or report['source_filename'] != filename:
    raise RuntimeError('Run cells 4 and 5 successfully before downloading.')
files.download(str(result_zip))

## Inspect another image
Rerun cell 4 to upload a new photo, then cells 5 and 6. To adjust masks for the current photo, rerun only cells 5 and 6.

The ZIP contains `overlay.png`, `mask.png` (class IDs 0–4), and `report.json`. A mask may look nearly black in an ordinary image viewer because its pixels store small class IDs. Coordinates refer to the EXIF-corrected image resized to at most 1600 pixels on its longest side.

## Troubleshooting
- Model download failed: check runtime internet access and rerun cell 3.
- Out of memory: try a smaller image or reconnect to a fresh runtime and rerun the notebook.
- Package imports fail after setup: restart the runtime and run from cell 2. If `torchvision::nms` appears, use a fresh runtime; this notebook does not require torchvision.
- Empty masks do not prove that the wire is normal. Inspect the original image and explore thresholds in cell 5.

[Colab upload/download documentation](https://colab.research.google.com/notebooks/io.ipynb) · [Pretrained model](https://huggingface.co/CIDAS/clipseg-rd64-refined)